<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>자료 기반 응답(Context-based Response)</strong>에 대해 학습합니다.
>PDF 문서를 로드하고 LLM에 컨텍스트로 주입하여 정확한 응답을 학습해봅시다.

# 문서 로드 (Document Loading)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">PyMuPDFLoader</mark>를 사용하여 PDF 파일을 Document 객체로 변환합니다.

LLM이 알고 있는 지식에는 두 가지 근본적인 공백이 있습니다. 첫째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 데이터 시점 한계</mark>로 인해 LLM은 특정 날짜까지의 데이터로만 학습되었기 때문에 그 이후에 작성된 보고서나 발표된 수치는 전혀 알지 못합니다. 둘째, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">조직 내부 문서 미포함</mark> 문제로 사내 보고서, 계약서, 내부 매뉴얼 등은 공개된 인터넷 데이터가 아니므로 LLM의 학습 데이터에 포함될 수 없습니다. 따라서 LLM이 정확하고 맥락에 맞는 답변을 하려면, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">질문과 관련된 자료를 직접 프롬프트에 담아 전달</mark>하는 방식이 필요합니다. 이것이 자료 기반 응답(Context-based Response)의 핵심 동기입니다.</br>

이 과정에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">문서 로더</mark>(PDF, Word 등 다양한 파일 형식을 텍스트로 변환하는 LangChain의 `DocumentLoader` 계열 도구), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트 분할</mark>(긴 문서를 LLM이 처리할 수 있는 크기로 나누는 청킹 개념), 그리고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM API</mark>(프롬프트를 전송하고 텍스트 응답을 받는 기본 호출 패턴)에 대한 이해가 필요합니다.

In [ ]:
# TODO 1: PDF 문서 로더로 "report.pdf"를 로드하여 documents에 저장하고, 페이지 수, 첫 페이지 내용(100자), 메타데이터를 출력해봅시다.

from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("report.pdf")
documents = loader.load()

# Document 구조 확인
print(f"페이지 수: {len(documents)}")
print(f"첫 페이지 내용 (100자): {documents[0].page_content[:100]}...")
print(f"메타데이터: {documents[0].metadata}")

## Document 객체 구조

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">속성</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>page_content</code></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">텍스트 내용</mark> (문자열)</td></tr>
    <tr><td style="text-align:center"><code>metadata</code></td><td>출처, 페이지 번호 등 부가 정보</td></tr>
  </tbody>
</table>

# 컨텍스트 주입 (Context Injection)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ChatPromptTemplate</mark>으로 문서 내용을 프롬프트에 삽입하여 LLM이 자료 기반으로 답변하도록 합니다.

In [ ]:
# TODO 2: 프롬프트 템플릿에 {context}와 {question} 플레이스홀더를 설정하고, documents의 텍스트 내용을 context로, "매출액은?"을 question으로 전달하여 체인을 실행해봅시다.

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
다음 자료를 기반으로 질문에 답변하세요.

자료:
{context}

질문: {question}
""")

# 문서 내용을 context로 전달
context = "\n".join([doc.page_content for doc in documents])
chain = prompt | llm
response = chain.invoke({"context": context, "question": "매출액은?"})
print(response.content)

## 자료 주입 전후 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">항목</th>
      <th>자료 없이</th>
      <th>자료 주입 후</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">정확성</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">환각 가능</mark></td><td>자료 기반 답변</td></tr>
    <tr><td style="text-align:center">출처</td><td>불명확</td><td>문서에서 추출</td></tr>
    <tr><td style="text-align:center">최신성</td><td>학습 데이터 한정</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실시간 문서 반영</mark></td></tr>
  </tbody>
</table>

💡ChatPromptTemplate 변수
> `{context}`와 `{question}`은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">플레이스홀더</mark>로, `invoke()` 시 딕셔너리로 값을 전달합니다.

💡문서가 너무 길면?
> LLM의 컨텍스트 윈도우에는 한계가 있습니다.
> 문서를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">청크(chunk)</mark>로 분할하고, 관련 청크만 검색하여 전달하는 것이 RAG의 핵심입니다.